# Character training data generator
This notebook generates 42x42px PNG images of characters for training data.

In [1]:
import string
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils import preprocessing_remove_lines, preprocessing_split_by_characters

In [2]:
keys = list(string.ascii_lowercase) + [str(i) for i in range(10)]
char_dict = {k: [] for k in keys}
char_dir = 'chars_train_data/'
# Delete all files in the target folder
for filename in tqdm(os.listdir(char_dir), desc="Deleting old files"):
    file_path = os.path.join(char_dir, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(".png"):
        os.remove(file_path)
        
os.makedirs(char_dir, exist_ok=True)

Deleting old files: 100%|█████████████████████████████████████████████████████| 43984/43984 [00:03<00:00, 12254.68it/s]


# Start character separation

In [3]:
wrong_count = 0
images_count = 0

# Loop through all png in train/
for (root,dirs,files) in os.walk('clean_train_data/',topdown=True):
    for file in tqdm(files, desc="Loading images"):
        if file.endswith('.png'):
            if file[1] == "_":
                os.remove(os.path.join(root, file))
                continue
            if file.split("-")[1].split(".png")[0] != "0":
                continue
                
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)
            processed_img = preprocessing_remove_lines(img) # Remove lines
            num_chars, char_images = preprocessing_split_by_characters(processed_img) # Split chars
            ground_truth = img_path.split("-")[0].split("/")[-1] # Get Captcha label

            # If wrong seperation count, do not add to chars_train_data
            images_count += 1
            if len(char_images) != len(ground_truth):
                wrong_count += 1
                continue

            # Correct seperated
            for i in range(len(ground_truth)):
                k = ground_truth[i]
                char_dict[k].append(char_images[i])

print(f"Total captcha count: {images_count}")
print(f"Wrong character sep count: {wrong_count}")
            

Loading images: 100%|██████████████████████████████████████████████████████████████| 8001/8001 [10:23<00:00, 12.83it/s]

Total captcha count: 7812
Wrong character sep count: 384


### Process character image after separation

In [4]:
def process_character_image(img):
    # --- Convert to grayscale ---
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # --- Invert so character = white (255), background = black (0) ---
    gray = 255 - gray
    
    # --- Threshold to make binary ---
    # _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # --- Find bounding box of character ---
    coords = cv2.findNonZero(gray)
    x, y, w, h = cv2.boundingRect(coords)
    
    # --- Crop + 3px padding ---
    pad = 3
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, gray.shape[1])
    y2 = min(y + h + pad, gray.shape[0])
    cropped = gray[y1:y2, x1:x2]

    # --- Normalize brightness: stretch intensity range to full 0–255 ---
    min_val, max_val = np.min(cropped), np.max(cropped)
    if max_val > min_val:  # avoid divide-by-zero if image is uniform
        cropped = (cropped - min_val) * (255.0 / (max_val - min_val))
        cropped = np.clip(cropped, 0, 255).astype(np.uint8)
    
    # --- Target canvas and padding ---
    target_size = 42
    pad = 3
    available_size = target_size - 2 * pad  # 38×38 drawable area

    h, w = cropped.shape
    scale = min(available_size / h, available_size / w)  # scale to fit within 38×38

    # --- Resize with preserved aspect ratio ---
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)

    # --- Create blank 42×42 black background ---
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    
    # --- Center the character ---
    y_offset = (target_size - new_h) // 2
    x_offset = (target_size - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

    return canvas

### Save to PNG training data

In [5]:
# Save each image to PNG
for k, img_list in char_dict.items():
    for idx, img in tqdm(enumerate(img_list), desc=k):
        filename = f"{k}_{idx:05d}.png"  # e.g., a_0000.png
        path = os.path.join(char_dir, filename)
        processed_img = process_character_image(img)
        cv2.imwrite(path, processed_img)



a: 1197it [00:03, 329.86it/s]
b: 1259it [00:04, 281.89it/s]
c: 1217it [00:04, 278.29it/s]
d: 1272it [00:04, 288.35it/s]
e: 1267it [00:04, 314.33it/s]
f: 1239it [00:03, 369.52it/s]
g: 1262it [00:03, 355.91it/s]
h: 1244it [00:03, 351.21it/s]
i: 1235it [00:03, 361.03it/s]
j: 1177it [00:03, 351.56it/s]
k: 1238it [00:03, 355.24it/s]
l: 1232it [00:03, 362.46it/s]
m: 1244it [00:03, 336.68it/s]
n: 1292it [00:03, 355.40it/s]
o: 1219it [00:03, 349.59it/s]
p: 1265it [00:03, 341.14it/s]
q: 1285it [00:03, 348.69it/s]
r: 1235it [00:03, 340.80it/s]
s: 1216it [00:03, 322.20it/s]
t: 1232it [00:03, 354.86it/s]
u: 1203it [00:03, 338.35it/s]
v: 1240it [00:05, 240.56it/s]
w: 1206it [00:03, 352.35it/s]
x: 1269it [00:03, 325.42it/s]
y: 1187it [00:03, 303.49it/s]
z: 1227it [00:04, 287.16it/s]
0: 1241it [00:04, 308.20it/s]
1: 1245it [00:03, 329.88it/s]
2: 1171it [00:04, 284.49it/s]
3: 1252it [00:04, 307.47it/s]
4: 1231it [00:03, 318.86it/s]
5: 1185it [00:03, 319.29it/s]
6: 1223it [00:03, 329.36it/s]
7: 1171it 